# Phase 4 EDA — Dataset #6 (in-flight emergencies) LEMD inspection

**Purpose.** Inspect OpenSky scientific Dataset #6 — the 832 flights that squawked **7700** (general emergency), 2018-01-01 to 2020-01-29 (Olive et al., *OpenSky Report 2020*) — to scope it as the **Layer 4 external-validation** set (D-008). We count the LEMD-associated subset, tabulate emergency categories, and decide how finely Layer 4 can slice.

**⚠️ FIREWALL.** This is *inspection only* (metadata: counts, categories, airports). It is Phase-4-legal. **Scoring these flights with the model is SEALED until Phase 7** (after model selection per D-006). Do NOT load these into any training/validation/threshold/selection step. Same posture as the test set.

**Cross-references**
- `backend/docs/ml/dataset6-emergency-external-validation.md` — the full exploration / acquisition spec (read this first).
- `backend/docs/ml/decisions/D-008-output-validation-layers.md` — the 5-layer validation stack; this is Layer 4.
- `backend/docs/ml/07-eval-prep.md` > "Layer 4" — the Phase-7 execution protocol + pre-committed finding template.
- `backend/docs/writeup/09-the-architectural-critique.md` > "External validation" — the narrative slot this fills.

**Environment note (2026-06-01).** The `traffic` library (dep, v2.13) is currently **broken in the backend venv** — it imports `DatetimeTZBlock` from a pandas internal path that newer pandas removed (`ImportError`). So `from traffic.data.datasets import squawk7700` fails today. This notebook works around it by reading the dataset's **metadata CSV directly from Zenodo** (record `3937483`), which needs only `pandas`. Fixing `traffic` (pin pandas / bump traffic) is a team-level dependency decision — see the env note at the bottom.

In [ ]:
import pandas as pd
from pathlib import Path

LEMD = "LEMD"  # Madrid-Barajas ICAO
# 200 km bbox around LEMD (same window as cycle 3) — used only in the trajectory pass (see end).
LEMD_REF = (40.4936, -3.5668)

# Local cache of the Zenodo metadata file (gitignored data/ dir).
CACHE = Path("../data/external/squawk7700_metadata.csv")
ZENODO_META = "https://zenodo.org/api/records/3937483/files/squawk7700_metadata.csv/content"

## 1. Load the metadata

If the cache is missing, download it once (Zenodo blocks default user-agents — send a browser UA). The 131 MB `squawk7700_trajectories.parquet.gz` is **not** needed here; it is only required for the sealed Phase-7 scoring pass.

In [ ]:
if not CACHE.exists():
    import urllib.request
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(ZENODO_META, headers={"User-Agent": "Mozilla/5.0"})
    CACHE.write_bytes(urllib.request.urlopen(req, timeout=120).read())
    print("downloaded ->", CACHE)

md = pd.read_csv(CACHE)
print("total 7700 flights (global):", len(md))
print("columns:", list(md.columns))

## 2. LEMD-association by airport code

We associate by **airport ICAO code** (`origin` / `destination` / `landing` / `diverted` == LEMD), **not** by Filter B/D. Filter B/D select for *normal LEMD operation geometry*; applying them to emergencies would discard exactly the trajectory-anomalous flights Layer 4 exists to test (circular). See the exploration doc §3.

In [ ]:
airport_cols = [c for c in ["origin", "destination", "landing", "diverted"] if c in md.columns]
mask = pd.Series(False, index=md.index)
for c in airport_cols:
    mask |= (md[c] == LEMD)
sub = md[mask].copy()

print(f"LEMD-associated flights (any of {airport_cols} == LEMD): {len(sub)}")
for c in airport_cols:
    print(f"  {c}==LEMD: {int((md[c]==LEMD).sum())}")

In [ ]:
show = [c for c in ["flight_id", "callsign", "typecode", "origin", "destination",
                     "landing", "diverted", "avh_problem"] if c in md.columns]
sub[show]

In [ ]:
# Relationship to LEMD (non-exclusive) + emergency category + aircraft type
arr = (sub["landing"] == LEMD) | (sub["destination"] == LEMD)
dep = (sub["origin"] == LEMD)
div = (sub["diverted"] == LEMD)
print("arriving/landing LEMD:", int(arr.sum()))
print("departing LEMD      :", int(dep.sum()))
print("diverted TO LEMD    :", int(div.sum()))
print("\navh_problem (LEMD subset):")
print(sub["avh_problem"].value_counts(dropna=False).to_string())
print("\ntypecode (LEMD subset):")
print(sub["typecode"].value_counts(dropna=False).to_string())

## 3. Findings (run 2026-06-01) and what they mean for Layer 4

**N = 6** LEMD-associated 7700 flights by airport code (of 832 global):

| flight_id | type | origin | dest | landing | diverted | avh_problem |
|---|---|---|---|---|---|---|
| IBK6241_20180616 | B738 | BIKF | LEMD | EGBB | EGBB | hydraulics |
| LAN706_20180918 | B789 | SCEL | LEMD | — | — | — |
| EZY42LB_20181213 | A319 | EGPH | LEMD | EGGD | EGGD | — |
| AFR11DN_20190816 | A321 | LFPG | LEMD | — | LFBD | cabin_pressure |
| BCS63A_20190829 | A306 | LEMD | EHAM | LEMD | LEMD | — |
| LCO1501_20191208 | B763 | LEMD | GVAC | — | — | — |

Relationship: 5 arriving/destined LEMD, 2 departing, 1 diverted-to-LEMD (BCS63A returned to LEMD). Only 2 of 6 carry an Aviation-Herald problem label (hydraulics, cabin pressure).

**Implications (this is the "settle once N is known" decision from the exploration doc §8):**

1. **N=6 is too small to stratify** by emergency category (§5 of the exploration doc) or to drop type-novel flights. Layer 4 at this N is a **small-N qualitative + non-parametric** signal (per-flight percentile + Mann-Whitney U with N reported), explicitly **not** a headline AUROC. The 07-eval-prep protocol already anticipates this — honor it.
2. **Airport-code association under-counts.** It misses 7700 flights that *transited / held in the LEMD TMA* but originated and landed elsewhere — which can be the most trajectory-anomalous cases. The **bbox-on-trajectories pass** (next step) will likely recover more. That pass needs the 131 MB trajectory parquet and is still Phase-4-legal (geometry only, no scoring).
3. **The Western-Europe fallback is now likely on the table.** If bbox recovery still leaves N in single digits, widen to a Western-European subset for statistical power — *with the explicit writeup caveat* that this conflates LEMD-specific signal with the broader manned-aviation distribution (07-eval-prep Layer 4 "Fallback").
4. **These 6 are mostly commercial jets on (probably) normal approaches** — a hydraulics/cabin-pressure event often still flies a near-standard arrival. Expect several to score ~normal, *correctly*, because the model scores trajectory shape, not transponder code. BCS63A (departed LEMD, returned to LEMD — a turn-back) is the most likely trajectory-anomalous case in this set; flag it for the qualitative review.

## 4. Next steps

**Phase 4 (still legal, recommended next):**
- Pull `squawk7700_trajectories.parquet.gz` (Zenodo, 131 MB) and run the **bbox pass**: keep flights with ≥ N points inside the 200 km LEMD box (recovers TMA-transit flights airport codes miss). Record the union count with the 6 above.
- For the union set, plot altitude-vs-time and 2D ground tracks; sanity-check they're in our area of competence. (Needs the trajectory file — not done here.)
- Write the final counts + this table into `backend/docs/ml/04-eda.md` and link the exploration doc.

**Phase 6:** leave Dataset #6 untouched. One-line reminder at the Phase 6 entry gate.

**Phase 7 (sealed until then):** resample 1 s → 10 s, run the locked model through the *same* preprocessing + train scaler, score once, report pooled percentile + Mann-Whitney U + per-flight qualitative notes; fill `[N]`/`[X]`/`[K]` in writeup 09.

---

### Environment fix needed (team decision)

`traffic` 2.13 fails to import in the backend venv (`ImportError: cannot import name 'DatetimeTZBlock' from 'pandas.core.internals.blocks'`) — a pandas-version incompatibility. Any notebook importing `traffic` is blocked until this is resolved (pin pandas to a `traffic`-compatible version, or bump `traffic`). Given the container pins versions and `uv.lock` is intentionally uncommitted, raise this with the team rather than churning deps locally. The metadata-direct path used here sidesteps it for Phase-4 inspection, but the Phase-7 trajectory pass will want a working `traffic` (or a direct parquet read).